# 365 Probabilidades — Dia #009
## Qual a probabilidade de você estar trabalhando horas que não produzem nada?

**Tipo:** Risco  
**Data de publicação:** 2026-06-22  
**Ferramenta:** Python  
**Decisão analisada:** Minhas horas extras realmente valem a pena?  
**Hashtag:** #365Probabilidades #Dia009

---

### 📖 A História

A Constituição brasileira permite 44 horas semanais de trabalho — mais 4 horas extras por semana. Quarenta e oito horas no total. É o teto legal.

E é exatamente onde a ciência diz que a produtividade começa a cair.

John Pencavel, economista de Stanford, estudou a relação entre horas trabalhadas e output real usando um dataset único: registros de produção de uma fábrica de munições durante a Primeira Guerra Mundial. Contexto onde a demanda era infinita, a pressão era máxima — e ainda assim havia um limite claro para o que o corpo humano conseguia entregar.

O resultado é chamado de efeito altamente não-linear: cada hora extra até 48h ainda adiciona alguma produção. A partir daí, o retorno começa a cair. Em 55 horas, o output é o mesmo que em 48. E quem trabalha 70 horas por semana produz exatamente o mesmo que quem trabalha 55.

Quinze horas desperdiçadas. Toda semana.

---

### 📚 O Conceito: Retorno Decrescente do Trabalho

O modelo de Pencavel demonstra que a relação entre horas trabalhadas e output não é linear — ela é côncava e, após um certo ponto, inverte. Isso acontece por três mecanismos combinados: fadiga cognitiva acumulada, aumento de erros e retrabalho, e declínio da capacidade de tomada de decisão.

A conexão com o Brasil: **48 horas por semana** é o teto constitucional (44h + 4h extras). É também o ponto onde Pencavel identifica o início da queda. Quem está dentro do limite legal já está no limiar do desperdício.

---

### 🧮 O Modelo
**Fontes:**
- Pencavel, J. (2014) — *The Productivity of Working Hours* — Stanford University / IZA Discussion Paper N=dados longitudinais de fábrica WWI
- PNAD Contínua (2023) — IBGE — N>200.000 trabalhadores brasileiros
- Melbourne Institute (2016) — *Working Hours and Cognitive Function* — N=6.000 trabalhadores australianos (+40 anos)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print('✅ Bibliotecas carregadas')

In [ ]:
# --- DADOS DA LITERATURA ---

# Pencavel (2014) — Stanford/IZA
# Output relativo por jornada semanal (base = 40h = 1.00)
horas      = np.array([35, 40, 44, 48, 50, 55, 60, 70])
output_rel = np.array([0.78, 1.00, 1.06, 1.09, 1.08, 1.09, 1.04, 1.09])

# Limiar constitucional brasileiro
h_limite_br = 48   # 44h + 4h extras

# Horas apos as quais output nao cresce mais
h_plateau   = 55   # output em 70h = output em 55h
h_desperd   = 70   # quem trabalha 70h produz igual a 55h
horas_zero  = h_desperd - h_plateau  # 15h desperdicadas/semana

# PNAD Continua 2023 — IBGE
# Proporcao de trabalhadores brasileiros com 49h ou mais
p_acima_48h = 0.25
n_pnad      = 200000

# Fator de correcao padrao do projeto
fator_correcao = 0.80

print('=' * 65)
print('  DADOS DA LITERATURA — PRODUTIVIDADE E HORAS DE TRABALHO')
print('=' * 65)
print(f'\n  Pencavel 2014 (Stanford/IZA):')
print(f'  → Queda de produtividade inicia em        : {h_limite_br}h/semana')
print(f'  → Output em 55h = output em               : {h_plateau}h/semana')
print(f'  → Output em 70h = output em               : {h_plateau}h/semana')
print(f'  → Horas desperdicadas (70h vs 55h)         : {horas_zero}h/semana')
print(f'  → Horas desperdicadas por ano             : {horas_zero * 52}h')
print(f'\n  PNAD Continua 2023 (IBGE — N>{n_pnad:,}):')
print(f'  → Trabalhadores brasileiros acima de 48h  : {p_acima_48h*100:.0f}%')
print(f'\n  Limite constitucional brasileiro         : {h_limite_br}h/semana')
print(f'  Fator de correcao aplicado                : x{fator_correcao}')
print('=' * 65)

In [ ]:
# --- O MODELO ---
# Distribuicao Beta para proporcao acima de 48h (PNAD)

alpha_m = p_acima_48h * n_pnad
beta_m  = (1 - p_acima_48h) * n_pnad
dist_m  = stats.beta(alpha_m, beta_m)
ic_m    = dist_m.interval(0.95)

p_corrigido = p_acima_48h * fator_correcao

# Perda de produtividade calculada
output_48h = output_rel[horas == 48][0]
output_55h = output_rel[horas == 55][0]
output_70h = output_rel[horas == 70][0]

perda_48_55 = (output_48h - output_55h) / output_48h * 100  # negativa = ganho real zero
horas_extras_ineficazes = h_desperd - h_plateau

print('=' * 65)
print('  MODELO — RESULTADO')
print('=' * 65)
print(f'  Trabalhadores acima de 48h — estimativa  : {p_acima_48h*100:.0f}%')
print(f'  Trabalhadores acima de 48h — IC 95%      : [{ic_m[0]*100:.2f}%, {ic_m[1]*100:.2f}%]')
print(f'  Trabalhadores acima de 48h — corrigido   : {p_corrigido*100:.1f}%')
print()
print(f'  Output em 48h (pico)                     : {output_48h:.2f}')
print(f'  Output em 55h (= 48h)                    : {output_55h:.2f}')
print(f'  Output em 70h (= 55h)                    : {output_70h:.2f}')
print()
print(f'  Horas extras ineficazes (70h vs 55h)     : {horas_extras_ineficazes}h/semana')
print(f'  Equivalente anual                        : {horas_extras_ineficazes * 52}h/ano')
print(f'  Equivalente em dias de trabalho           : {horas_extras_ineficazes * 52 / 8:.0f} dias/ano')
print('=' * 65)

In [ ]:
# --- VISUALIZACAO ---

# GRAFICO 1 — Curva de output por horas trabalhadas (Pencavel)
fig1, ax1 = plt.subplots(figsize=(12, 8))

horas_cont = np.linspace(35, 72, 200)
# Interpolacao suave baseada nos pontos de Pencavel
from scipy.interpolate import PchipInterpolator
interp = PchipInterpolator(horas, output_rel)
output_cont = interp(horas_cont)

ax1.plot(horas_cont, output_cont, color='#c0392b', linewidth=3)
ax1.fill_between(horas_cont, output_cont, alpha=0.1, color='#c0392b')

# Linha de limite constitucional brasileiro
ax1.axvline(x=48, color='#c8a84b', linestyle='--', linewidth=2,
            label='Limite constitucional BR: 48h/semana')

# Zona de desperdicio
ax1.axvspan(55, 70, alpha=0.08, color='#c0392b', label='Zona de desperdicio: 55h-70h = mesmo output')

# Marcadores
ax1.scatter(horas, output_rel, color='#c0392b', s=60, zorder=5)
for h, o in zip([44, 48, 55, 70], [1.06, 1.09, 1.09, 1.09]):
    ax1.annotate(f'{h}h\n({o:.2f})', xy=(h, o), xytext=(0, 12),
                 textcoords='offset points', ha='center', fontsize=9,
                 color='#333333', fontweight='bold')

ax1.set_xlabel('Horas trabalhadas por semana', fontsize=12)
ax1.set_ylabel('Output relativo (base = 40h = 1.00)', fontsize=12)
ax1.set_title('Curva de produtividade por horas trabalhadas — Efeito Nao-Linear de Pencavel\nPencavel (2014) — Stanford/IZA',
              fontsize=13, pad=15)
ax1.legend(fontsize=11)
ax1.set_xlim(34, 73)

plt.tight_layout()
plt.savefig('dia-009-grafico-01-curva.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Grafico 1 salvo!')


# GRAFICO 2 — Horas desperdicadas por ano
fig2, ax2 = plt.subplots(figsize=(12, 8))

cenarios = ['40h/semana\n(referencia)', '48h/semana\n(limite BR)', '55h/semana\n(plateau)', '70h/semana\n(comum)']
outputs  = [1.00, 1.09, 1.09, 1.09]
cores2   = ['#2a8a82', '#c8a84b', '#c8a84b', '#c0392b']

bars = ax2.bar(cenarios, outputs, color=cores2, alpha=0.85, width=0.5)
ax2.set_ylim(0.90, 1.15)
ax2.set_ylabel('Output total relativo', fontsize=12)
ax2.set_title('Output identico a partir de 55h — 15 horas extras por semana desperdicadas\nPencavel (2014) — Stanford/IZA',
              fontsize=13, pad=15)

for bar, o in zip(bars, outputs):
    ax2.text(bar.get_x() + bar.get_width() / 2, o + 0.004,
             f'{o:.2f}', ha='center', fontweight='bold', fontsize=13)

# Anotacao de desperdicio
ax2.annotate('', xy=(3, 1.09), xytext=(2, 1.09),
             arrowprops=dict(arrowstyle='<->', color='#c0392b', lw=2))
ax2.text(2.5, 1.095, '15h extras\nnenhum ganho', ha='center',
         fontsize=10, color='#c0392b', fontweight='bold')

plt.tight_layout()
plt.savefig('dia-009-grafico-02-desperdicio.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Grafico 2 salvo!')


# GRAFICO 3 — Distribuicao Beta (proporcao acima de 48h)
fig3, ax3 = plt.subplots(figsize=(12, 8))

x = np.linspace(ic_m[0] * 0.999, ic_m[1] * 1.001, 1000)
y = dist_m.pdf(x)

ax3.plot(x * 100, y, color='#c0392b', linewidth=3)
ax3.fill_between(x * 100, y, alpha=0.2, color='#c0392b')
ax3.axvline(x=ic_m[0] * 100, color='#c8a84b', linestyle='--', linewidth=2,
            label=f'IC 95%: [{ic_m[0]*100:.2f}%, {ic_m[1]*100:.2f}%]')
ax3.axvline(x=ic_m[1] * 100, color='#c8a84b', linestyle='--', linewidth=2)
ax3.axvline(x=p_acima_48h * 100, color='#c0392b', linewidth=2,
            label=f'Estimativa central: {p_acima_48h*100:.0f}%')
ax3.axvline(x=p_corrigido * 100, color='#2a8a82', linestyle='-.',
            linewidth=2, label=f'Corrigido (x0.80): {p_corrigido*100:.1f}%')

ax3.set_xlabel('Proporcao de trabalhadores brasileiros acima de 48h/semana (%)', fontsize=12)
ax3.set_ylabel('Densidade', fontsize=12)
ax3.set_title('Distribuicao Beta — Trabalhadores brasileiros acima do limite de produtividade\nPNAD Continua 2023 (IBGE) — N>200.000',
              fontsize=13, pad=15)
ax3.legend(fontsize=11)

plt.tight_layout()
plt.savefig('dia-009-grafico-03-bayesiano.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Grafico 3 salvo!')

### 💡 O Insight

**25% dos trabalhadores brasileiros ultrapassam 48 horas semanais** — exatamente o limiar onde a ciência diz que a produtividade começa a cair. IC 95%: [24,81%, 25,19%].

E o número mais revelador: quem trabalha **70 horas por semana produz o mesmo que quem trabalha 55**. Quinze horas extras por semana — 780 horas por ano, quase 98 dias de trabalho — que não adicionam absolutamente nada ao output.

O limite constitucional brasileiro não é apenas uma proteção trabalhista. É, por acidente, também o limiar da eficiência humana.

*Quantas das suas horas extras da última semana realmente produziram algo?*

---

### ⚠️ Limitações do Modelo
- O estudo de Pencavel foi conduzido com trabalhadores de uma fábrica de munições na WWI — contexto de trabalho físico e repetitivo, que pode não generalizar diretamente para trabalho cognitivo ou criativo do século XXI
- A relação output x horas pode variar significativamente por tipo de trabalho, autonomia e motivação
- Os dados da PNAD capturam horas declaradas — podem subestimar horas reais em contextos de trabalho remoto e disponibilidade via WhatsApp
- O fator de correção (×0,80) é uma aproximação padrão do projeto

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*